In [4]:
from dataset_processing import (
    BIODIVNER_DIR, IBMCCNER_DIR, transform_to_ner_format, 
    ner_dataset_to_hf_format, generate_consistent_label_map, 
    IBMCCNER_LABELS, BIODIVNER_LABELS, CLIMATEIE_LABELS, CLIRENER_LABELS_V1, 
    biodivner_process_bio_documents, shorten_name, 
    ibmccner_process_bio_documents, cwed4eta_process_json_file)
from EXPERIMENTS.calculate_hf_dataset_stats import process_dataset_split, plot_class_distribution
import pandas as pd
import os
from datasets import load_dataset, concatenate_datasets


Helper functions:

In [32]:
import numpy as np
from collections import Counter
import math

def calculate_advanced_metrics(dataset_split, label_names):
    """
    Calculates advanced content-based metrics for NER datasets.
    Normalizes metrics by token count rather than document count to handle
    variable document sizes (sentences vs full papers).
    """
    
    # --- Accumulators ---
    total_tokens = 0
    total_docs = 0
    
    # Entity Accumulators
    all_entity_texts = []    # List of strings, e.g., ["carbon dioxide", "Paris", ...]
    all_span_lengths = []    # List of integers
    class_counts = Counter()
    
    # Structure Accumulators
    total_entities = 0
    adjacency_hits = 0       # Entities immediately following another
    
    # Pre-check
    if len(dataset_split) == 0:
        return None

    for row in dataset_split:
        tokens = row['tokens']
        tags = row['ner_tags']
        
        doc_len = len(tokens)
        total_tokens += doc_len
        total_docs += 1
        
        # --- 1. Extract Entities from BIO Tags ---
        # We need the exact span indices to check adjacency
        current_entities = [] # Stores (start_index, end_index, label_name, text)
        
        i = 0
        while i < len(tags):
            tag_id = tags[i]
            
            # Skip invalid tags
            if tag_id < 0 or tag_id >= len(label_names):
                i += 1
                continue
                
            tag_name = label_names[tag_id]
            
            if tag_name.startswith("B-"):
                start_idx = i
                class_type = tag_name[2:]
                
                # Look ahead for I- tags
                j = i + 1
                while j < len(tags):
                    next_tag_id = tags[j]
                    if next_tag_id < 0 or next_tag_id >= len(label_names):
                        break
                    next_tag_name = label_names[next_tag_id]
                    if next_tag_name == f"I-{class_type}":
                        j += 1
                    else:
                        break
                end_idx = j # Exclusive
                
                # Capture Entity Details
                span_text = " ".join(tokens[start_idx:end_idx])
                span_len = end_idx - start_idx
                
                current_entities.append({
                    "start": start_idx,
                    "end": end_idx,
                    "label": class_type,
                    "text": span_text,
                    "len": span_len
                })
                
                # Update Loop index
                i = j 
            else:
                i += 1

        # --- 2. Update Aggregates ---
        for ent in current_entities:
            all_entity_texts.append(ent['text'])
            all_span_lengths.append(ent['len'])
            class_counts[ent['label']] += 1
            
        total_entities += len(current_entities)

        # --- 3. Calculate Adjacency (Clumping) for this Doc ---
        # We check if Entity B starts within 1 token of Entity A ending.
        # This catches "A, B" (sep by comma) or "A B" (immediate).
        for k in range(len(current_entities) - 1):
            prev_end = current_entities[k]['end']
            next_start = current_entities[k+1]['start']
            
            # Gap of 0 means immediate (end 5, start 5). 
            # Gap of 1 means one separator (end 5, start 6 -> token 5 is sep).
            gap = next_start - prev_end
            if gap <= 1: 
                adjacency_hits += 1

    # --- 4. Final Calculations ---
    
    # A. Saturation (Density)
    # Entities per 100 tokens. Comparable across doc sizes.
    saturation = (total_entities / total_tokens) * 100 if total_tokens > 0 else 0
    
    # B. Vocabulary (TTR)
    # Are we seeing new things, or the same 5 words repeated?
    # Lowercase to ensure "Carbon" and "carbon" count as same type.
    unique_entities = set(t.lower() for t in all_entity_texts)
    vocab_size = len(unique_entities)
    ttr = (vocab_size / total_entities) if total_entities > 0 else 0
    
    # C. Complexity (Long Spans)
    # Percentage of entities longer than 4 tokens
    long_spans = sum(1 for l in all_span_lengths if l >= 5)
    long_span_ratio = (long_spans / total_entities) * 100 if total_entities > 0 else 0
    
    # D. Structure (Adjacency)
    # What % of entities appear in a list/cluster?
    adjacency_ratio = (adjacency_hits / total_entities) * 100 if total_entities > 0 else 0

    # E. Class Balance (Normalized Entropy)
    # 0 = All same class, 1 = Perfectly balanced
    probs = [count / total_entities for count in class_counts.values()]
    if len(probs) > 1:
        entropy = -sum(p * math.log2(p) for p in probs)
        max_entropy = math.log2(len(probs))
        normalized_entropy = entropy / max_entropy
    else:
        normalized_entropy = 0.0

    return {
        "Total Tokens": total_tokens,
        "Total Entities": total_entities,
        "Saturation (%)": saturation,
        "Entity TTR (0-1)": ttr,
        "Avg Span Len": np.mean(all_span_lengths) if all_span_lengths else 0,
        "Long Spans (>4t) %": long_span_ratio,
        "Clumping (Adj) %": adjacency_ratio,
        "Class Entropy (0-1)": normalized_entropy
    }
    
def print_comparison(stats_dict):
    """Helper to pretty print the new stats"""
    header = f"{'Metric':<25} | {'Value':<10}"
    print("-" * 40)
    print(header)
    print("-" * 40)
    for k, v in stats_dict.items():
        if isinstance(v, float):
            print(f"{k:<25} | {v:<10.2f}")
        else:
            print(f"{k:<25} | {v:<10}")
    print("-" * 40)
    
    
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

def plot_per_class_ttr(dataset_split, label_names, split_name, output_dir):
    """
    Calculates TTR (Unique / Total) per entity class and plots it.
    High TTR = Diverse vocabulary (e.g., many different locations).
    Low TTR = Repetitive vocabulary (e.g., same 5 chemicals repeated).
    """
    
    # --- 1. Accumulate Texts per Class ---
    # Dictionary to store list of texts for each label: {'LOC': ['Paris', 'London', 'Paris'], ...}
    class_texts = defaultdict(list)
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["mathtext.fontset"] = "dejavuserif"
    
    for row in dataset_split:
        tokens = row['tokens']
        tags = row['ner_tags']
        
        i = 0
        while i < len(tags):
            tag_id = tags[i]
            if tag_id < 0 or tag_id >= len(label_names):
                i += 1
                continue
            
            tag_name = label_names[tag_id]
            
            # Found start of entity
            if tag_name.startswith("B-"):
                label_type = tag_name[2:] # Remove "B-"
                start_idx = i
                
                # Find the end
                j = i + 1
                while j < len(tags):
                    next_tag_id = tags[j]
                    if next_tag_id < 0 or next_tag_id >= len(label_names):
                        break
                    next_tag_name = label_names[next_tag_id]
                    if next_tag_name == f"I-{label_type}":
                        j += 1
                    else:
                        break
                
                # Extract text and store
                span_text = " ".join(tokens[start_idx:j])
                class_texts[label_type].append(span_text)
                
                i = j 
            else:
                i += 1

    # --- 2. Calculate TTR ---
    ttr_data = []
    
    print(f"\n--- Per-Class TTR ({split_name}) ---")
    for label, texts in class_texts.items():
        total_count = len(texts)
        # Lowercase to ensure "Carbon" and "carbon" count as the same type
        unique_count = len(set(t.lower() for t in texts))
        
        if total_count > 0:
            ttr = unique_count / total_count
            ttr_data.append({
                "Label": label, 
                "TTR": ttr, 
                "Count": total_count,
                "Unique": unique_count
            })
            print(f"{label:<15} | TTR: {ttr:.2f} | Unique: {unique_count}/{total_count}")

    # Sort by TTR descending for better visualization
    ttr_data.sort(key=lambda x: x['Label'], reverse=False)
    
    if not ttr_data:
        print("No entities found to plot.")
        return

    # --- 3. Plotting ---
    plt.figure(figsize=(10, 6))
    
    # Create Bar Plot
    labels = [d['Label'] for d in ttr_data]
    values = [d['TTR'] for d in ttr_data]
    bar_width = 0.65
    
    # Use a color palette
    # colors = sns.color_palette("viridis", len(labels))
    bars = plt.bar(labels, values, color="#8C2981", edgecolor='black', width=bar_width)
    
    
    # Aesthetics
    # plt.title(f"Entity Diversity (TTR) by Class - {split_name.title()}", fontsize=14)
    # plt.ylabel("Type-Token Ratio (0.0 = Repetitive, 1.0 = Unique)", fontsize=12)
    # plt.xlabel("Entity Label", fontsize=12)
    plt.ylim(0, 1.1) # Give space for text labels
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.xticks(rotation=45, fontsize=8, ha='right')
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                 f'{height:.2f}',
                 ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    
    # Save
    filename = f"plot_ttr_per_class_{split_name}.png"
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"TTR Plot saved to {save_path}")

## BioDivNER

In [42]:
print("Converting from BIO tags to char spans.")
split_name = "overall"
hf_name, safe_hf_name, output_dir = "BioDivNER", "BioDivNER", "PLOTS"


biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=BIODIVNER_DIR, 
    labels_to_keep=BIODIVNER_LABELS, 
    split=["train", "validation", "test"]
    )

transformed_dataset, tag_map = transform_to_ner_format(biodivner_structured_data, BIODIVNER_LABELS)

dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

new_stats = calculate_advanced_metrics(dataset_train["train"], list(generate_consistent_label_map(BIODIVNER_LABELS).keys()))
print_comparison(new_stats)

stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(BIODIVNER_LABELS).keys()))

header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10} | {'Sentences':<10}"
print("\n" + header)
print("-" * len(header))
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10} | "
        f"{int(stats['Total Sentences']):<10}")


# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)



Converting from BIO tags to char spans.
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/test.csv...
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/dev.csv...
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/train.csv...
----------------------------------------
Metric                    | Value     
----------------------------------------
Total Tokens              | 114824    
Total Entities            | 10010     
Saturation (%)            | 8.72      
Entity TTR (0-1)          | 0.16      
Avg Span Len              | 1.50      
Long Spans (>4t) %        | 0.60      
Clumping (Adj) %          | 26.28     
Class Entropy (0-1)       | 0.86      
----------------------------------------

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities   | Sentences 
--------------------------------------------------------------------------------------------------------
overall      | 47

## IBMCCNER

In [41]:
print("Converting from BIO tags to char spans.")
split_name = "overall"
hf_name, safe_hf_name, output_dir = "ibm-research/Climate-Change-NER", shorten_name(hf_name), "PLOTS"
split = ["train", "validation", "test"]

print(f"Loading '{IBMCCNER_DIR}' with splits: {split}")
ds = load_dataset(IBMCCNER_DIR) 
train_documentwise = []
temp_list = []
if type(split) == type(["list"]):
    for sp in split:
        for line in ds[sp]["text"]:
            if line.strip().startswith('-DOCSTART-'):
                if temp_list:
                    train_documentwise.append(temp_list)
                temp_list = []
            temp_list.append(line)
        if temp_list:
            train_documentwise.append(temp_list)
else:
    for line in ds[split]["text"]:
        if line.strip().startswith('-DOCSTART-'):
            if temp_list:
                train_documentwise.append(temp_list)
            temp_list = []
        temp_list.append(line)
    if temp_list:
        train_documentwise.append(temp_list)

print("Converting from BIO tags to char spans.")        
structured_documents_char_spans = ibmccner_process_bio_documents(
    document_list=train_documentwise,
    labels_to_keep=IBMCCNER_LABELS
)

transformed_dataset, tag_map = transform_to_ner_format(structured_documents_char_spans, IBMCCNER_LABELS)

dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

new_stats = calculate_advanced_metrics(dataset_train["train"], list(generate_consistent_label_map(IBMCCNER_LABELS).keys()))
print_comparison(new_stats)

stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(IBMCCNER_LABELS).keys()))

header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10} | {'Sentences':<10}"
print("\n" + header)
print("-" * len(header))
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10} | "
      f"{int(stats['Total Sentences']):<10}")


# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)

Converting from BIO tags to char spans.
Loading 'ibm-research/Climate-Change-NER' with splits: ['train', 'validation', 'test']
Converting from BIO tags to char spans.
----------------------------------------
Metric                    | Value     
----------------------------------------
Total Tokens              | 44690     
Total Entities            | 4270      
Saturation (%)            | 9.55      
Entity TTR (0-1)          | 0.55      
Avg Span Len              | 1.84      
Long Spans (>4t) %        | 5.20      
Clumping (Adj) %          | 24.24     
Class Entropy (0-1)       | 0.90      
----------------------------------------

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities   | Sentences 
--------------------------------------------------------------------------------------------------------
overall      | 85.29      | 8.15       | 9.20       | 10.47      | 1.84       | 4270       | 524       


## ClimateIE

In [40]:
DATA_DIRECTORY = "DATA/ClimateIE/human_corpus/"
print("Converting from BIO tags to char spans.")
split_name = "overall"
hf_name, safe_hf_name, output_dir = "Jo-Pan/ClimateIE", shorten_name(hf_name), "PLOTS"
split = ["train", "validation", "test"]


# Note: SpaCy is no longer strictly needed for splitting, 
# but I'll leave the import commented out in case you need it later for word tokenization.
# import spacy 
# nlp = spacy.load("en_core_sci_sm", disable=["ner"])

try:
    file_names = [f for f in os.listdir(DATA_DIRECTORY) if f.endswith('.json')]
except FileNotFoundError:
    print(f"Error: The directory '{DATA_DIRECTORY}' was not found.")
    file_names = []

# This list will hold the final transformed data for all files
structured_documents_char_spans = []

# Process each file
for i, file_name in enumerate(file_names):
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    
    with open(file_path, "r", encoding="utf-8") as f:
        document = json.load(f)
    
    # Raw text and global entities dictionary
    raw_text = document["text"]
    raw_entities = document["entities"] # This is a dict of ID -> {begin, end, ...}

    # Convert the entities dict to a list
    # Since we are not splitting by sentence, we do NOT need to recalculate offsets.
    valid_entities = []
    
    for ent_id, ent_data in raw_entities.items():
        valid_entities.append({
            "text": ent_data['substring'],
            "label": ent_data['label'],
            "start": ent_data['begin'], # Global start (no math needed)
            "end": ent_data['end'],     # Global end (no math needed)
            "identifier": ent_data.get('identifier', None)
        })

    # Sort entities by start position (good practice)
    valid_entities.sort(key=lambda x: x['start'])

    # Append the whole document as one unit
    structured_documents_char_spans.append({
        "id": file_name,            # Tracking source file
        "text": raw_text,           # The full document text
        "sentence_id": i,           # Using file index as ID since we don't have sentences
        "entities": valid_entities
    })

# Output verification
print(f"Processing complete. Total documents collected: {len(structured_documents_char_spans)}")


transformed_dataset, tag_map = transform_to_ner_format(structured_documents_char_spans, CLIMATEIE_LABELS)

dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

new_stats = calculate_advanced_metrics(dataset_train["train"], list(generate_consistent_label_map(CLIMATEIE_LABELS).keys()))
print_comparison(new_stats)

stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(CLIMATEIE_LABELS).keys()))

header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10} | {'Sentences':<10}"
print("\n" + header)
print("-" * len(header))
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10} | "
      f"{int(stats['Total Sentences']):<10}")


# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)

Converting from BIO tags to char spans.
Processing complete. Total documents collected: 25
----------------------------------------
Metric                    | Value     
----------------------------------------
Total Tokens              | 226812    
Total Entities            | 13015     
Saturation (%)            | 5.74      
Entity TTR (0-1)          | 0.23      
Avg Span Len              | 1.65      
Long Spans (>4t) %        | 1.56      
Clumping (Adj) %          | 15.41     
Class Entropy (0-1)       | 0.69      
----------------------------------------

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities   | Sentences 
--------------------------------------------------------------------------------------------------------
overall      | 9072.48    | 520.60     | 16.00      | 17.43      | 1.65       | 13015      | 25        


In [37]:
DATA_DIRECTORY = "DATA/ClimateIE/human_corpus/"
print("Converting from BIO tags to char spans.")
split_name = "overall"
hf_name, safe_hf_name, output_dir = "Jo-Pan/ClimateIE", shorten_name(hf_name), "PLOTS"
split = ["train", "validation", "test"]
import spacy

# 1. Setup SpaCy
print("Importing SpaCy ...")
# Using sci_sm as requested, disabling ner for speed since we have our own labels
nlp = spacy.load("en_core_sci_sm", disable=["ner"])

try:
    file_names = [f for f in os.listdir(DATA_DIRECTORY) if f.endswith('.json')]
except FileNotFoundError:
    print(f"Error: The directory '{DATA_DIRECTORY}' was not found.")
    file_names = []

# This list will hold the final transformed data for all files
structured_documents_char_spans = []

# Process each file
for file_name in file_names:
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    
    with open(file_path, "r", encoding="utf-8") as f:
        document = json.load(f)
    
    # Raw text and global entities dictionary
    raw_text = document["text"]
    raw_entities = document["entities"] # This is a dict of ID -> {begin, end, ...}

    # 2. Tokenize into sentences
    # Increasing max_length in case documents are very large
    nlp.max_length = len(raw_text) + 100000 
    doc = nlp(raw_text)

    # Convert the entities dict to a list for easier iteration
    # We store the ID inside the dict for reference
    entity_list = []
    for ent_id, ent_data in raw_entities.items():
        ent_data['id_key'] = ent_id # Keep the original key just in case
        entity_list.append(ent_data)

    # Sort entities by start position to potentially optimize or ensure order
    entity_list.sort(key=lambda x: x['begin'])

    sentence_counter = 0

    # Iterate over sentences found by SpaCy
    for sent in doc.sents:
        sent_text = sent.text
        # These are the global offsets of the sentence within the document
        sent_start_char = sent.start_char
        sent_end_char = sent.end_char
        
        valid_local_entities = []

        # 3. Remap entities to sentences
        for ent in entity_list:
            # Check if the entity is strictly contained within this sentence
            if ent['begin'] >= sent_start_char and ent['end'] <= sent_end_char:
                
                # Calculate new offsets relative to the sentence
                local_start = ent['begin'] - sent_start_char
                local_end = ent['end'] - sent_start_char
                
                # 4. Transform data format (including Identifier)
                valid_local_entities.append({
                    "text": ent['substring'],
                    "label": ent['label'],
                    "start": local_start,
                    "end": local_end,
                    "identifier": ent.get('identifier', None) # Bonus: Keep identifier
                })

        # Only add sentences that have text (skip empty newlines)
        if sent_text.strip():
            structured_documents_char_spans.append({
                "text": sent_text,
                "sentence": sent_text, # Redundant but matches your requested format
                "sentence_id": sentence_counter,
                "entities": valid_local_entities
            })
            sentence_counter += 1

# Output verification
print(f"Processing complete. Total sentences collected: {len(structured_documents_char_spans)}")

transformed_dataset, tag_map = transform_to_ner_format(structured_documents_char_spans, CLIMATEIE_LABELS)

dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(CLIMATEIE_LABELS).keys()))

header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10} | {'Sentences':<10}"
print("\n" + header)
print("-" * len(header))
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10} | "
      f"{int(stats['Total Sentences']):<10}")


# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)


Converting from BIO tags to char spans.
Importing SpaCy ...


/home/p0l3/miniforge3/envs/clirener_finetune/lib/python3.10/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Processing complete. Total sentences collected: 6552

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities   | Sentences 
--------------------------------------------------------------------------------------------------------
overall      | 34.62      | 1.99       | 16.11      | 17.43      | 1.65       | 13014      | 6552      


## CliReNER

In [33]:
print("Converting from BIO tags to char spans.")
split_name = "overall"
hf_name = "P0L3/CliReNER_v_1_1_28_GOLD_authorannots"
safe_hf_name, output_dir = shorten_name(hf_name), "PLOTS"
split = ["train", "validation", "test"]

print(f"Loading dataset: {hf_name}...")
try:
    hf_dataset = load_dataset(hf_name)
except Exception as e:
    print(f"Error loading dataset: {e}")

# --- 2. Data Preparation ---
# Get Label Names from the 'train' split
if "train" in hf_dataset:
    tags_feature = hf_dataset["train"].features["ner_tags"].feature
else:
    # Fallback if train doesn't exist, check other splits
    available_split = list(hf_dataset.keys())[0]
    tags_feature = hf_dataset[available_split].features["ner_tags"].feature
    
label_names = tags_feature.names

# Create Overall Dataset
datasets_list = [hf_dataset[split] for split in hf_dataset.keys()]
combined_dataset = concatenate_datasets(datasets_list)

# Define splits dictionary
splits_to_process = {}
for split in hf_dataset.keys():
    splits_to_process[split] = hf_dataset[split]
splits_to_process["overall"] = combined_dataset

# --- 3. Calculation Loop ---
# Updated Header to show both ratios
header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10}"
print("\n" + header)
print("-" * len(header))

all_stats_data = []

for split_name, dataset_obj in splits_to_process.items():
    # Calculate Stats
    stats, counts = process_dataset_split(dataset_obj, label_names)
    
    if stats is None:
        print(f"Skipping empty split: {split_name}")
        continue
    
    if split_name == "overall":
        new_stats = calculate_advanced_metrics(dataset_obj, label_names)
        print_comparison(new_stats)
        plot_per_class_ttr(dataset_obj, label_names, split_name, output_dir)

    # Print to Console
    print(f"{split_name:<12} | "
            f"{stats['Sentence Length (Mean)']:<10.2f} | "
            f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
            f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
            f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
            f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
            f"{int(stats['Total Entities']):<10}")
    
    # Generate Plot
    plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)
    
    # Collect Data for CSV
    csv_row = {"Split": split_name}
    csv_row.update(stats)
    all_stats_data.append(csv_row)

print("-" * len(header))

# --- 4. Save to CSV ---
csv_filename = f"{safe_hf_name}_statistics.csv"
csv_filepath = os.path.join(output_dir, csv_filename)

df_stats = pd.DataFrame(all_stats_data)

# Reorder columns for readability in CSV
cols = [
    "Split", 
    "Sentence Length (Mean)", 
    "Sentence Length (Median)", 
    "Entity Density (Avg entities/sent)", 
    "Ratio (Median Len / Density)",
    "Ratio (Mean Len / Density)",
    "Span Length (Avg tokens/entity)", 
    "Total Entities", 
    "Total Sentences"
]

# Ensure we only select columns that exist
cols = [c for c in cols if c in df_stats.columns]
df_stats = df_stats[cols]

df_stats.to_csv(csv_filepath, index=False)
print(f"\nStatistics saved to CSV: {csv_filepath}")
print(f"Plots saved to directory: {output_dir}/")

Converting from BIO tags to char spans.
Loading dataset: P0L3/CliReNER_v_1_1_28_GOLD_authorannots...

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities  
-------------------------------------------------------------------------------------------
train        | 48.84      | 11.78      | 3.74       | 4.15       | 1.90       | 1802      
validation   | 59.57      | 14.81      | 3.11       | 4.02       | 1.97       | 311       
test         | 55.11      | 14.11      | 3.97       | 3.91       | 1.67       | 254       
----------------------------------------
Metric                    | Value     
----------------------------------------
Total Tokens              | 9715      
Total Entities            | 2367      
Saturation (%)            | 24.36     
Entity TTR (0-1)          | 0.74      
Avg Span Len              | 1.88      
Long Spans (>4t) %        | 4.69      
Clumping (Adj) %          | 49.51     
Class Entropy (0-1)       | 0.94      
--------

In [ ]:
from random import sample


['Meteorological Phenomenon',
 'Natural Disaster',
 'Body of Water',
 'Satellite',
 'Quantity',
 'Method',
 'Physical Artefact',
 'Natural Phenomenon',
 'Measuring Device',
 'Energy Source',
 'Geographical Feature',
 'Organization',
 'Field of Study',
 'Policy']

In [86]:
lsfile_path = "DATA/LABEL_STUDIO/project-30-at-2025-11-14-12-19-2a7464a5.json"
data = cwed4eta_process_json_file(lsfile_path)

hf_name, safe_hf_name, output_dir = "P0L3/CliReNER_v_1_1_28_FULLtest", shorten_name(hf_name), "PLOTS"


print("Transforming data to BIO tags ...")
transformed_dataset, tag_map = transform_to_ner_format(data, sample(list(CLIRENER_LABELS_V1), len(CLIRENER_LABELS_V1)//2))

print("Transforming data to HF Dataset ...")
hf_dataset = ner_dataset_to_hf_format(transformed_dataset, tag_map)

# --- 2. Data Preparation ---
# Get Label Names from the 'train' split
if "train" in hf_dataset:
    tags_feature = hf_dataset["train"].features["ner_tags"].feature
else:
    # Fallback if train doesn't exist, check other splits
    available_split = list(hf_dataset.keys())[0]
    tags_feature = hf_dataset[available_split].features["ner_tags"].feature
    
label_names = tags_feature.names

# Create Overall Dataset
datasets_list = [hf_dataset[split] for split in hf_dataset.keys()]
combined_dataset = concatenate_datasets(datasets_list)

# Define splits dictionary
splits_to_process = {}
for split in hf_dataset.keys():
    splits_to_process[split] = hf_dataset[split]
splits_to_process["overall"] = combined_dataset

# --- 3. Calculation Loop ---
# Updated Header to show both ratios
header = f"{'Split':<12} | {'Len(Mean)':<10} | {'Density':<10} | {'Ratio(Med)':<10} | {'Ratio(Mean)':<10} | {'Span Len':<10} | {'Entities':<10}"
print("\n" + header)
print("-" * len(header))

all_stats_data = []

for split_name, dataset_obj in splits_to_process.items():
    # Calculate Stats
    stats, counts = process_dataset_split(dataset_obj, label_names)
    
    if stats is None:
        print(f"Skipping empty split: {split_name}")
        continue
    
    if split_name == "overall":
        new_stats = calculate_advanced_metrics(dataset_obj, label_names)
        print_comparison(new_stats)

    # Print to Console
    print(f"{split_name:<12} | "
            f"{stats['Sentence Length (Mean)']:<10.2f} | "
            f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
            f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
            f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
            f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
            f"{int(stats['Total Entities']):<10}")
    
#     # Generate Plot
#     plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)
    
#     # Collect Data for CSV
#     csv_row = {"Split": split_name}
#     csv_row.update(stats)
#     all_stats_data.append(csv_row)

# print("-" * len(header))

# # --- 4. Save to CSV ---
# csv_filename = f"{safe_hf_name}_statistics.csv"
# csv_filepath = os.path.join(output_dir, csv_filename)

# df_stats = pd.DataFrame(all_stats_data)

# # Reorder columns for readability in CSV
# cols = [
#     "Split", 
#     "Sentence Length (Mean)", 
#     "Sentence Length (Median)", 
#     "Entity Density (Avg entities/sent)", 
#     "Ratio (Median Len / Density)",
#     "Ratio (Mean Len / Density)",
#     "Span Length (Avg tokens/entity)", 
#     "Total Entities", 
#     "Total Sentences"
# ]

# # Ensure we only select columns that exist
# cols = [c for c in cols if c in df_stats.columns]
# df_stats = df_stats[cols]

# df_stats.to_csv(csv_filepath, index=False)
# print(f"\nStatistics saved to CSV: {csv_filepath}")
# print(f"Plots saved to directory: {output_dir}/")

Transforming data to BIO tags ...
Transforming data to HF Dataset ...

Split        | Len(Mean)  | Density    | Ratio(Med) | Ratio(Mean) | Span Len   | Entities  
-------------------------------------------------------------------------------------------
train        | 34.00      | 3.87       | 8.01       | 8.78       | 1.74       | 3793      
validation   | 37.55      | 3.94       | 8.88       | 9.52       | 1.69       | 485       
test         | 33.58      | 3.76       | 8.11       | 8.93       | 1.81       | 459       
----------------------------------------
Metric                    | Value     
----------------------------------------
Total Tokens              | 42033     
Total Entities            | 4737      
Saturation (%)            | 11.27     
Entity TTR (0-1)          | 0.56      
Avg Span Len              | 1.74      
Long Spans (>4t) %        | 2.05      
Clumping (Adj) %          | 24.28     
Class Entropy (0-1)       | 0.91      
---------------------------------------

1228